In [2]:
# Install the python library
!pip install pysr

# Import and install the required Julia backend
# (This might take a few minutes the first time you run it)
import pysr
pysr.install()

In [ ]:
import numpy as np
import pandas as pd
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
import warnings
import time

# Suppress PySR warnings
warnings.filterwarnings("ignore")


# Data Generation
def count_real_roots(coeffs):
    if coeffs[0] == 0: return -1 
    try:
        roots = np.roots(coeffs)
        real_roots = roots[np.isreal(roots)]
        return len(real_roots)
    except:
        return -1

def generate_polynomial_data(degree, n_samples=5000, seed=None):
    if seed is not None:
        np.random.seed(seed)
    X = np.random.uniform(-10, 10, (n_samples, degree + 1))
    y = np.apply_along_axis(count_real_roots, 1, X)
    valid_idx = y != -1
    return X[valid_idx], y[valid_idx]


# Experiment Runner
def run_full_robustness_test():
    # Config
    experiments = [
        {'degree': 2, 'time': 60,  'desc': 'Quadratic'},
        {'degree': 3, 'time': 120, 'desc': 'Cubic'},
        {'degree': 4, 'time': 180, 'desc': 'Quartic'},
        {'degree': 5, 'time': 300, 'desc': 'Quintic'}
    ]
    
    n_trials = 1
    summary_stats = []
    
    print("="*80)
    print(f" (Running {n_trials} trials per Degree)")
    print("="*80)
    
    for exp in experiments:
        deg = exp['degree']
        print(f"\n--- Testing Degree {deg} ({exp['desc']}) ---")
        
        trial_accuracies = []
        best_eq_of_all = ""
        best_acc_of_all = 0.0
        
        # Fix data
        X, y = generate_polynomial_data(deg, n_samples=5000, seed=42)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        var_names = ['a', 'b', 'c', 'd', 'e', 'f'][:deg+1]
        
        for i in range(n_trials):
            start_time = time.time()
            print(f"  Trial {i+1}/{n_trials}...", end=" ")
            
            # New random state for the model evolution
            model = PySRRegressor(
                niterations=200, 
                binary_operators=["+", "-", "*"], 
                unary_operators=["square", "abs", "sign"], 
                timeout_in_seconds=exp['time'],
                maxsize=45,
                population_size=2000,
                model_selection="best", 
                loss="L2DistLoss()",
                parsimony=0.001,
                verbosity=0,
                random_state=i*100 # Distinct seed
            )
            
            model.fit(X_train, y_train, variable_names=var_names)
            
            # Evaluate
            preds = model.predict(X_test)
            y_pred = np.clip(np.round(preds), 0, deg)
            acc = balanced_accuracy_score(y_test, y_pred)
            trial_accuracies.append(acc)
            
            # Save best equation
            if acc > best_acc_of_all:
                best_acc_of_all = acc
                best_eq_of_all = model.sympy()
            
            print(f"Acc: {acc:.1%} ({int(time.time() - start_time)}s)")
            
        # Stats
        mean_acc = np.mean(trial_accuracies)
        std_acc = np.std(trial_accuracies)
        summary_stats.append({
            'Degree': deg,
            'Mean Acc': mean_acc,
            'Std Dev': std_acc,
            'Best Acc': best_acc_of_all,
            'Best Equation': str(best_eq_of_all)
        })
        
    return summary_stats
        
stats = run_full_robustness_test()

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
 (Running 1 trials per Degree)

--- Testing Degree 2 (Quadratic) ---
  Trial 1/1... 